In [ ]:
import networkx as nx
import numpy as np
import datetime
import math
import torch
from torch import nn
from torch.nn import Module, Parameter
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import pickle
import time
from google.colab import files

In [ ]:
opt = {}
opt["dataset"] = "diginetica_bins" # 
opt["batchSize"] = 100
opt["hiddenSize"] = 100
opt["epoch"] = 30
opt["lr"] = 0.001
opt["lr_dc"] = 0.1
opt["lr_dc_step"] = 3
opt["l2"] = 1e-5
opt["step"] = 1
opt["patience"] = 3
opt["nonhybrid"] = True
opt["validation"] = True
opt["valid_portion"] = 0.1
opt["seed"] = 40
opt["n_trials"] = 3
opt["n_node"] = 43098 #37484 For Yoochoose,  43098 For Diginetica, 60,417 for Nowplaying

In [ ]:
def data_masks(all_usr_pois, item_tail):
    us_lens = [len(upois) for upois in all_usr_pois] # len of each session
    len_max = max(us_lens)
    us_pois = [upois + item_tail * (len_max - le) for upois, le in zip(all_usr_pois, us_lens)] # Set each session to len equals "len_max" and use 0s as right padding
    us_msks = [[1] * le + [0] * (len_max - le) for le in us_lens] # mask to let us know the actual values and the ceros
    return us_pois, us_msks, len_max

In [ ]:
def split_validation(train_set, valid_portion, seed):
    train_set_x, train_set_y, train_set_t = train_set
    n_samples = len(train_set_x)

    # Set seed for reproducibility
    rng = np.random.RandomState(seed)
    sidx = np.arange(n_samples, dtype='int32')
    rng.shuffle(sidx)

    n_train = int(np.round(n_samples * (1. - valid_portion)))

    # 1. Validation split
    valid_set_x = [train_set_x[s] for s in sidx[n_train:]]
    valid_set_y = [train_set_y[s] for s in sidx[n_train:]]
    valid_set_t = [train_set_t[s] for s in sidx[n_train:]]

    # 2. Training split
    train_set_x = [train_set_x[s] for s in sidx[:n_train]]
    train_set_y = [train_set_y[s] for s in sidx[:n_train]]
    train_set_t = [train_set_t[s] for s in sidx[:n_train]]

    # Return two 3-element tuples
    return (train_set_x, train_set_y, train_set_t), (valid_set_x, valid_set_y, valid_set_t)

In [ ]:
class Data():
    def __init__(self, data, shuffle=False, graph=None):
        inputs = data[0]
        times = data[2]
        times, _, _ = data_masks(times, [0])
        self.times = np.asarray(times)
        inputs, mask, len_max = data_masks(inputs, [0]) # padding
        self.inputs = np.asarray(inputs) # sessions of len equals "len_max"
        self.mask = np.asarray(mask)
        self.len_max = len_max
        self.targets = np.asarray(data[1])
        self.length = len(inputs)
        self.shuffle = shuffle
        self.graph = graph

    def generate_batch(self, batch_size):
        if self.shuffle:
            shuffled_arg = np.arange(self.length)
            np.random.shuffle(shuffled_arg)
            self.inputs = self.inputs[shuffled_arg]
            self.mask = self.mask[shuffled_arg]
            self.targets = self.targets[shuffled_arg]
            self.times = self.times[shuffled_arg] # new addition
        n_batch = int(self.length / batch_size)
        if self.length % batch_size != 0:
            n_batch += 1
        slices = np.split(np.arange(n_batch * batch_size), n_batch)
        slices[-1] = slices[-1][:(self.length - batch_size * (n_batch - 1))]
        return slices

    def get_slice(self, i):
      inputs, mask, targets, times = self.inputs[i], self.mask[i], self.targets[i], self.times[i]
      items, n_node, A, B, alias_inputs = [], [], [], [], []
      for u_input in inputs:
        n_node.append(len(np.unique(u_input)))
      max_n_node = np.max(n_node)

      def build_time_matrix(inputs, times):
        batch_matrices = []
        max_session_len = max([len(s) for s in inputs])

        for u_input, u_times in zip(inputs, times):
          u_t = np.array(u_times)

          # Obtenemos la longitud real de esta sesión específica
          real_len = sum(1 for item in u_input if item != 0)
          if real_len == 0:
            real_len = len(u_input) # Por si no hay padding aún

          diff = np.abs(u_t[:real_len, None] - u_t[None, :real_len])

          diff_log = np.log1p(diff)

          padded_matrix = np.zeros((max_session_len, max_session_len))
          padded_matrix[:real_len, :real_len] = diff_log

          batch_matrices.append(padded_matrix)

        return batch_matrices

      t_matrix = build_time_matrix(inputs, times)

      for u_input, u_times in zip(inputs, times):
        node = np.unique(u_input)
        items.append(node.tolist() + (max_n_node - len(node)) * [0])
        u_A = np.zeros((max_n_node, max_n_node))
        n_A = np.zeros((max_n_node, max_n_node))
        for i in np.arange(len(u_input) - 1):
          if u_input[i + 1] == 0:
              break
          u = np.where(node == u_input[i])[0][0]
          v = np.where(node == u_input[i + 1])[0][0]
          u_A[u][v] = 1
          n_A[u][v] = 1
          n_A[v][u] = 1

        # Matrix normalization
        u_sum_in = np.sum(u_A, 0)
        u_sum_in[np.where(u_sum_in == 0)] = 1
        u_A_in = np.divide(u_A, u_sum_in)
        u_sum_out = np.sum(u_A, 1)
        u_sum_out[np.where(u_sum_out == 0)] = 1
        u_A_out = np.divide(u_A.transpose(), u_sum_out)
        u_A = np.concatenate([u_A_in, u_A_out]).transpose()
        A.append(u_A)
        B.append(n_A)
        alias_inputs.append([np.where(node == i)[0][0] for i in u_input])
      return alias_inputs, A, B, items, mask, targets, torch.tensor(times), t_matrix

In [ ]:
class EnhancedGNN(nn.Module):
    def __init__(self, latent_size, dropout=0.5):
        super(EnhancedGNN, self).__init__()
        self.latent_size = latent_size

        self.linear_msg = nn.Sequential(
            nn.Linear(latent_size, latent_size),
            nn.LeakyReLU(0.2),
            nn.Dropout(dropout)
        )

        # Gate g1 (Update Gate)
        self.linear_gate = nn.Linear(2 * latent_size, latent_size)

        self.norm = nn.LayerNorm(latent_size)

    def forward(self, A, v):
        # 1. Agregación y Transformación
        a_in = torch.matmul(A, v)
        a_in = self.linear_msg(a_in)

        # 2. Gate Calculation
        combined = torch.cat([a_in, v], dim=-1)
        g1 = torch.sigmoid(self.linear_gate(combined))

        # 3. Gated Update con Residual Connection
        v_next = g1 * a_in + (1 - g1) * v

        # 4. Normalización final
        return self.norm(v_next)

In [ ]:
class NewGNN(nn.Module):
  def __init__(self, latent_size):
    super(NewGNN, self).__init__()
    self.latent_size = latent_size

    self.linear_msg = nn.Linear(latent_size, latent_size)

    self.linear_gate = nn.Linear(2 * latent_size, latent_size)

  def forward(self, A, v):
    """
    v: Hidden states [batch_size, n_nodes, latent_size]
    A: Adjacency matrix [batch_size, n_nodes, n_nodes]
    """
    # --- Equation 1: Message Propagation ---
    # A * v aggregates neighbor states: [batch, n, n] x [batch, n, d] -> [batch, n, d]
    a_in = torch.matmul(A, v)
    # Apply the H2 transformation and b2 bias
    a_in = self.linear_msg(a_in)

    # --- Equation 3: Gate Calculation (g1) ---
    # Concatenate the incoming message and the current node state
    combined = torch.cat([a_in, v], dim=-1)
    g1 = torch.sigmoid(self.linear_gate(combined))

    # --- Equation 2: Gated Update ---
    # Linearly interpolate between the message and the old state
    # v_next = g1 * message + (1 - g1) * old_state
    v_next = g1 * a_in + (1 - g1) * v

    return v_next

In [ ]:
class GGNN(Module):
    def __init__(self, hidden_size, step=1):
        super(GGNN, self).__init__()
        self.step = step
        self.hidden_size = hidden_size
        self.input_size = 3 * hidden_size
        self.gate_size = 3 * hidden_size

        self.b_iah = Parameter(torch.Tensor(self.hidden_size))
        self.b_oah = Parameter(torch.Tensor(self.hidden_size))

        self.w_z = Parameter(torch.Tensor(self.input_size, self.hidden_size))
        self.w_r = Parameter(torch.Tensor(self.input_size, self.hidden_size))
        self.w_h = Parameter(torch.Tensor(self.input_size, self.hidden_size))

        self.b_z = Parameter(torch.Tensor(self.hidden_size))
        self.b_r = Parameter(torch.Tensor(self.hidden_size))
        self.b_h = Parameter(torch.Tensor(self.hidden_size))

        self.linear_edge_in = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_out = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_f = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        #Parameters for time dynamics (delta)
        self.tanh = nn.Tanh()
        self.sigmoid = nn.Sigmoid()

    def GNNCell(self, A, hidden, times, prev_hidden=None, gnn=None):
        #Standard GGNN Message Passing
        input_in = torch.matmul(A[:, :, :A.shape[1]], self.linear_edge_in(hidden)) + self.b_iah
        input_out = torch.matmul(A[:, :, A.shape[1]: 2 * A.shape[1]], self.linear_edge_out(hidden)) + self.b_oah
        inputs = torch.cat([input_in, input_out], 2)
        combined = torch.cat([inputs, hidden], -1)

        # Update (GRU-based)
        z_s = self.sigmoid(combined @ self.w_z + self.b_z) # [batch_size x hidden_size]
        r_s = self.sigmoid(combined @ self.w_r + self.b_r) # [batch_size x hidden_size]
        h_c = self.tanh(torch.cat([inputs, hidden * r_s], -1) @ self.w_h + self.b_h) # [batch_size x hidden_size]
        h_y = z_s * hidden + (1 - z_s) * h_c
        return h_y

    def forward(self, A, hidden, times, gnn):
      for i in range(self.step):
          hidden = self.GNNCell(A, hidden, times, None, gnn)
      return hidden

In [ ]:
class SessionGraph(Module):
    def __init__(self, opt, n_node, max_session_len):
      super(SessionGraph, self).__init__()
      self.hidden_size = opt["hiddenSize"]
      self.n_node = n_node
      self.max_s_len = max_session_len
      self.batch_size = opt["batchSize"]
      self.nonhybrid = opt["nonhybrid"]
      self.embedding = nn.Embedding(self.n_node, self.hidden_size)
      self.gnn = EnhancedGNN(self.hidden_size)
      self.ggnn = GGNN(self.hidden_size, step=opt["step"])
      ############################################################################################################
      self.proj_q = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
      self.proj_k = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
      # self.time_decay_scale = nn.Parameter(torch.Tensor([0.1]))
      self.linear_transform = nn.Linear(self.hidden_size * 2, self.hidden_size, bias=True)
      self.loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
      self.optimizer = torch.optim.Adam(self.parameters(), lr=opt["lr"], weight_decay=opt["l2"])
      self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt["lr_dc_step"], gamma=opt["lr_dc"])
      self.linear = nn.Linear(self.hidden_size * 2, self.hidden_size, bias=True)
      # Parameters for attention mechanism
      self.dropout_p = 0.1 # Opcional: pasar vía opt
      self.attn_dropout = nn.Dropout(self.dropout_p)

      # 1. Parámetros para el componente Temporal (delta) -> L x L
      self.w_delta = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))
      self.b_delta = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))

      # 2. Parámetros para el componente de Contenido (tao) -> H x L
      self.w_tao = nn.Parameter(torch.Tensor(self.hidden_size, self.max_s_len))
      self.b_tao = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))

      # 3. Parámetros para la mezcla final de la compuerta (g) -> L x L
      self.w_g_delta = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))
      self.w_g_tao = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))
      self.b_g = nn.Parameter(torch.Tensor(self.max_s_len, self.max_s_len))
      self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def t_aware_mechanism(self, hidden_unique, original_mask, alias_inputs, time_aware=False, t_matrix_seq=None):
      batch_size = hidden_unique.shape[0]
      L = original_mask.shape[1] # Longitud L de la secuencia
      device = hidden_unique.device

      # =====================================================================
      # DESENROLLAR LA SECUENCIA: [B, N, H] -> [B, L, H]
      # =====================================================================
      batch_idx = torch.arange(batch_size).long().unsqueeze(1).to(device)
      seq_hidden = hidden_unique[batch_idx, alias_inputs]

      # =====================================================================
      # AUTO-ATENCIÓN SOBRE LA SECUENCIA: [B, L, L]
      # =====================================================================
      Q = self.proj_q(seq_hidden)
      K = self.proj_k(seq_hidden)
      raw_scores = torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(self.hidden_size)

      if time_aware:
        # Recortamos al tamaño L del batch actual
        w_delta = self.w_delta[:L, :L]
        b_delta = self.b_delta[:L, :L]

        w_tao = self.w_tao[:, :L]
        b_tao = self.b_tao[:L, :L]

        w_g_delta = self.w_g_delta[:L, :L]
        w_g_tao = self.w_g_tao[:L, :L]
        b_g = self.b_g[:L, :L]

        if not torch.is_tensor(t_matrix_seq):
            t_matrix_seq = torch.tensor(np.array(t_matrix_seq), device=device).float()

        # A. Delta (Tiempo): [B, L, L] * [L, L] -> [B, L, L]
        delta = torch.tanh(t_matrix_seq * w_delta + b_delta)

        # B. Tao (Contenido, Info Estructural): [B, L, H] @ [H, L] -> [B, L, L]
        tao = torch.tanh(torch.matmul(seq_hidden, w_tao) + b_tao)

        # C. Compuerta: [B, L, L]
        g = torch.sigmoid(delta * w_g_delta + tao * w_g_tao + b_g)

        raw_scores = raw_scores * g

      # Máscara para ignorar el padding de la secuencia original
      mask_expanded = (original_mask == 0).unsqueeze(1)
      raw_scores = raw_scores.masked_fill(mask_expanded, -1e9)

      # Softmax y vector de contexto actualizado
      attn_weights = F.softmax(raw_scores, dim=-1)
      updated_seq = torch.matmul(attn_weights, seq_hidden) # [B, L, H]
      updated_seq = seq_hidden + self.attn_dropout(updated_seq)

      last_item_idx = torch.sum(original_mask, 1) - 1
      ht = updated_seq[batch_idx.squeeze(-1), last_item_idx]

      if not self.nonhybrid:
          mask_float = original_mask.unsqueeze(-1).float()
          global_rep = torch.sum(updated_seq * mask_float, 1) / torch.sum(mask_float, 1)
          sf = self.linear_transform(torch.cat([global_rep, ht], 1))
      else:
          sf = ht

      Candidates = self.embedding.weight[1:]
      scores = torch.matmul(sf, Candidates.transpose(1, 0))

      return scores

    def forward(self, inputs, A, B, times):
      hidden = self.embedding(inputs)
      hidden_se = self.ggnn(A, hidden, times, self.gnn) + hidden
      hidden_st = self.gnn(B, hidden) + hidden
      return hidden_se, hidden_st, hidden

In [ ]:
def trans_to_cuda(variable):
    if torch.cuda.is_available():
        return variable.cuda()
    else:
        return variable


def trans_to_cpu(variable):
    if torch.cuda.is_available():
        return variable.cpu()
    else:
        return variable

def forward(model, i, data, plot=False):
  alias_inputs, A, B, items, mask, targets, deltas, t_ = data.get_slice(i)
  A = np.array(A)
  B = np.array(B)
  alias_inputs = trans_to_cuda(torch.Tensor(alias_inputs).long())
  items = trans_to_cuda(torch.Tensor(items).long())
  deltas = trans_to_cuda(torch.Tensor(deltas).float())
  A = trans_to_cuda(torch.Tensor(A).float())
  B = trans_to_cuda(torch.Tensor(B).float())
  mask = trans_to_cuda(torch.Tensor(mask).long())
  hidden_se, hidden_st, hidden = model(items, A, B, deltas)

  get_se = lambda i: hidden_se[i][alias_inputs[i]]
  get_st = lambda i: hidden_st[i][alias_inputs[i]]
  get_tm = lambda i: deltas[i][alias_inputs[i]]

  seq_hidden_se = torch.stack([get_se(i) for i in torch.arange(len(alias_inputs)).long()])
  seq_hidden_st = torch.stack([get_st(i) for i in torch.arange(len(alias_inputs)).long()])
  seq_hidden_tm = torch.stack([get_tm(i) for i in torch.arange(len(alias_inputs)).long()])

  scores_sec = model.t_aware_mechanism(hidden_se, mask, alias_inputs, False, t_)
  scores_non = model.t_aware_mechanism(hidden_st, mask, alias_inputs, False, t_)
  return targets, scores_sec + scores_non

In [ ]:
precision_values = []
mrr_values = []
training_times = []
throughput = []
full_training_times = []
filename = "yoo_a"
weights_filename = f'weights_{filename}.pt'

In [ ]:
def train_test(model, train_data, test_data):
    model.scheduler.step()
    print('start training: ', datetime.datetime.now())
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets - 1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting: ', datetime.datetime.now())
    model.eval()
    hit, mrr = [], []
    # --- Start Timing Throughput ---
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_predict = time.time()
    total_samples = 0

    slices = test_data.generate_batch(model.batch_size)
    for i in slices:
        targets, scores = forward(model, i, test_data, plot=False)
        total_samples += len(targets) # Count how many items were processed
        sub_scores = scores.topk(20)[1]
        sub_scores = trans_to_cpu(sub_scores).detach().numpy()
        for score, target, mask in zip(sub_scores, targets, test_data.mask):
            hit.append(np.isin(target - 1, score))
            if len(np.where(score == target - 1)[0]) == 0:
                mrr.append(0)
            else:
                mrr.append(1 / (np.where(score == target - 1)[0][0] + 1))

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_predict = time.time()
    
    prediction_time = end_predict - start_predict
    pps = total_samples / prediction_time if prediction_time > 0 else 0
    throughput.append(pps)

    hit = np.mean(hit) * 100
    mrr = np.mean(mrr) * 100

    print(f"\tInference Time: {prediction_time:.2f}s")
    print(f"\tThroughput:     {pps:.2f} predictions/sec")
    return hit, mrr, total_loss.item()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Después de montar Google Drive, puedes navegar por tus archivos. Aquí te muestro cómo listar el contenido de la raíz de tu Drive para empezar:

In [ ]:
import os
os.listdir('/content/drive/My Drive/Tesis/data/yoochoose_1_64_weights_pandas_attention')

['train.txt', 'test.txt', 'all_train_seq.txt']

In [ ]:
data_path = "/content/drive/My Drive/Tesis/data/"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device", device)

Device cuda


In [ ]:
train_data = pickle.load(open(data_path + opt["dataset"] + '/train.txt', 'rb'))
test_data = pickle.load(open(data_path + opt["dataset"] + '/test.txt', 'rb'))
test_data = Data(test_data, shuffle=False)

In [ ]:
def set_seed(seed):
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed) # multi-GPU

In [ ]:
def evaluate(model, eval_data):
    model.eval()
    hit, mrr = [], []
    slices = eval_data.generate_batch(model.batch_size)
    for i in slices:
        targets, scores = forward(model, i, eval_data, plot=False)
        sub_scores = scores.topk(20)[1]
        sub_scores = trans_to_cpu(sub_scores).detach().numpy()
        for score, target, mask in zip(sub_scores, targets, eval_data.mask):
            hit.append(np.isin(target - 1, score))
            if len(np.where(score == target - 1)[0]) == 0:
                mrr.append(0)
            else:
                mrr.append(1 / (np.where(score == target - 1)[0][0] + 1))
    return np.mean(hit) * 100, np.mean(mrr) * 100

In [ ]:
all_trials_history = []
for t in range(opt["n_trials"]):
    current_seed = opt["seed"] + t
    set_seed(current_seed)

    trial_loss = []
    trial_precision = []
    trial_mrr = []

    print(f"\n=======================================================")
    print(f"Experiment {t+1}/{opt['n_trials']} using seed: {current_seed}")

    # 1. PREPARAR DATOS DESDE CERO PARA CADA PRUEBA
    if opt["validation"]:
        print(f"Setting up data using a validation portion of {opt['valid_portion']} ...")
        tra_split, val_split = split_validation(train_data, opt["valid_portion"], current_seed)

        train_d = Data(tra_split, shuffle=True)
        valid_d = Data(val_split, shuffle=False)
    else:
        train_d = Data(train_data, shuffle=True)
        valid_d = test_data

    model = trans_to_cuda(SessionGraph(opt, opt["n_node"], 200))

    print("Starting Training")
    best_mrr = 0
    best_precision = 0
    best_epoch_mrr = 0
    best_epoch_precision = 0
    bad_counter = 0
    best_alpha = 0.0
    training_start = time.time()

    for epoch in range(opt["epoch"]):
        print('-------------------------------------------------------')
        print('epoch: ', epoch)
        epoch_start = time.time()

        precision, mrr, total_loss = train_test(model, train_d, valid_d)

        epoch_end = time.time()
        training_times.append(epoch_end - epoch_start)

        trial_loss.append(total_loss)
        trial_precision.append(precision)
        trial_mrr.append(mrr)

        flag = 0

        if precision > best_precision:
            best_precision = precision
            best_epoch_precision = epoch
            flag = 1
            torch.save(model.state_dict(), f'{weights_filename}_{t}.pt')

        if mrr > best_mrr:
            best_mrr = mrr
            best_epoch_mrr = epoch
            flag = 1
            torch.save(model.state_dict(), f'{weights_filename}_{t}.pt')

        print("training_time %f s" % (epoch_end - epoch_start))
        print('Best Validation Result So Far:')
        print('\tPrecision@20:\t%.4f\tMMR@20:\t%.4f\tEpoch:\t%d / %d' % (best_precision, best_mrr, epoch, opt["epoch"]))

        if flag == 1:
            bad_counter = 0
        else:
            bad_counter += 1

        if bad_counter >= opt["patience"]:
            print(f"Program stopped by patience parameter after {epoch + 1} epochs (no improvement for {opt['patience']} consecutive epochs)")
            break

    for i in range(len(trial_loss)):
        all_trials_history.append({
            'trial': t + 1,
            'epoch': i,
            'loss': trial_loss[i],
            'precision': trial_precision[i],
            'mrr': trial_mrr[i]
        })

    training_end = time.time()
    full_training_time = training_end - training_start
    full_training_times.append(full_training_time)

    print(f"\n--- Final Test Evaluation for Trial {t+1} ---")
    if opt["validation"]:
        # Cargar los pesos que guardamos en la mejor época
        model.load_state_dict(torch.load(f'{weights_filename}_{t}.pt'))

    final_test_precision, final_test_mrr = evaluate(model, test_data)
    print(f"Total training time: {training_end:.4f}")
    print(f"Final Test Precision@20: {final_test_precision:.4f}")
    print(f"Final Test MRR@20: {final_test_mrr:.4f}")

    precision_values.append(final_test_precision)
    mrr_values.append(final_test_mrr)

# --- REPORTE ESTADÍSTICO FINAL ---
precision_values = np.array(precision_values)
mrr_values = np.array(mrr_values)

print('\n' + '='*55)
print('FINAL RESULTS ACROSS ALL TRIALS')
print('='*55)
print(f"Average total training time: {np.mean(np.array(full_training_times)):.4f}s")
print(f"Average training time per epoch: {np.mean(np.array(training_times)):.4f}s")
print(f"Average throughput: {np.mean(np.array(throughput)):.4f} predictions per second")
print(f"Precision@20: {np.mean(precision_values):.4f} ± {np.std(precision_values):.4f}")
print(f"MRR@20:       {np.mean(mrr_values):.4f} ± {np.std(mrr_values):.4f}")


Experiment 1/3 using seed: 40
Setting up data using a validation portion of 0.1 ...
Starting Training
-------------------------------------------------------
epoch:  0
start training:  2026-05-09 20:30:30.770275


/tmp/ipykernel_2292/2683792196.py:2: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  model.scheduler.step()


[0/6476] Loss: 10.5181
[1296/6476] Loss: 7.6361
[2592/6476] Loss: 6.7715
[3888/6476] Loss: 6.3678
[5184/6476] Loss: 6.7278


/tmp/ipykernel_2292/2683792196.py:17: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('\tLoss:\t%.3f' % total_loss)


	Loss:	45936.785
start predicting:  2026-05-09 20:35:53.173673
	Inference Time: 36.73s
	Throughput:     1958.92 predictions/sec
training_time 359.829288 s
Best Validation Result So Far:
	Precision@20:	52.2343	MMR@20:	17.6804	Epoch:	0 / 30
-------------------------------------------------------
epoch:  1
start training:  2026-05-09 20:36:30.664238
[0/6476] Loss: 5.6496
[1296/6476] Loss: 5.8893
[2592/6476] Loss: 5.9053
[3888/6476] Loss: 5.9898
[5184/6476] Loss: 5.8172
	Loss:	38158.125
start predicting:  2026-05-09 20:41:47.251200
	Inference Time: 36.01s
	Throughput:     1997.72 predictions/sec
training_time 353.311234 s
Best Validation Result So Far:
	Precision@20:	53.9397	MMR@20:	18.5050	Epoch:	1 / 30
-------------------------------------------------------
epoch:  2
start training:  2026-05-09 20:42:24.057362
[0/6476] Loss: 5.6921
[1296/6476] Loss: 5.2941
[2592/6476] Loss: 5.4781
[3888/6476] Loss: 5.5143
[5184/6476] Loss: 5.2052
	Loss:	34180.969
start predicting:  2026-05-09 20:47:42.58

Test model using weights

In [ ]:
model = trans_to_cuda(SessionGraph(opt, opt["n_node"], 200))

state_dict = torch.load('weights_yoo_a.pt_0.pt', map_location=lambda storage, loc: storage)
model.load_state_dict(state_dict)

model.eval()

final_test_precision, final_test_mrr = evaluate(model, test_data)

print(f"Loaded Weights Results:")
print(f"Precision@20: {final_test_precision:.4f}")
print(f"MRR@20: {final_test_mrr:.4f}")

Loaded Weights Results:
Precision@20: 53.0201
MRR@20: 18.5978


Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plot_thesis_results(csv_file):
    df = pd.read_csv(csv_file)
    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    sns.lineplot(data=df, x='epoch', y='loss', ax=ax1,
                 color='#2c3e50', linewidth=2.5, marker='o', label='Mean Loss')
    ax1.set_title('Convergencia de la Función de Pérdida', fontsize=15, fontweight='bold')
    ax1.set_xlabel('Época', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)

    sns.lineplot(data=df, x='epoch', y='precision', ax=ax2,
                 color='#e67e22', linewidth=2.5, marker='s', label='Precision@20')
    sns.lineplot(data=df, x='epoch', y='mrr', ax=ax2,
                 color='#27ae60', linewidth=2.5, marker='D', label='MRR@20')

    ax2.set_title('Evolución de Métricas en Validación', fontsize=15, fontweight='bold')
    ax2.set_xlabel('Época', fontsize=12)
    ax2.set_ylabel('Score (%)', fontsize=12)
    ax2.legend(frameon=True, facecolor='white')

    plt.tight_layout()
    plt.savefig('learning_curves_thesis.png', dpi=300)
    plt.show()

plot_thesis_results('training_history.csv')